# Reproducing Armstrong & Vlasov (2026) from dandiset 001341

**Paper.** Armstrong AG, Vlasov Y (2026). Neural correlates of perceptual decision-making in the primary
somatosensory cortex. *PNAS* 123(18), e2514107123.
[doi:10.1073/pnas.2514107123](https://doi.org/10.1073/pnas.2514107123). The claims file was written from the
accepted author manuscript in PubMed Central (NIHMS2207436, PMC13142948) and its SI Appendix.

**Dandiset.** [DANDI:001341](https://dandiarchive.org/dandiset/001341/0.260414.1316), version 0.260414.1316. The
paper cites this version (reference 63, "Version 0.260414.1316", DOI 10.48324/dandi.001341/0.260414.1316), and it is
the only published version; the draft has the same two assets. The files are streamed; nothing is downloaded.

**What the dandiset holds.** Two NWB files, animals 489 and 503 of the 19 in Table S1, each with the units table,
the trials table and the ball tracking at 500 Hz. It holds no wall positions, whisker tracking, touch times, gait
or psychometric sessions, and no synchronisation signal between the spike times and the trial clock.

**Claims.** From `claims/001341/10.1073-pnas.2514107123.yaml`. Five are tested; the others are blocked.

| Claim | Tested here | Reason if blocked |
|---|---|---|
| table-s1-recordings | yes | |
| fig1lm-staged-modulation | no | metadata_insufficient: spikes cannot be placed on the trial clock |
| fig1n-epoch-rates | no | metadata_insufficient (as above) |
| fig1jk-all-units | no | metadata_insufficient (as above) |
| fig1n-ramp-slope | no | metadata_insufficient (as above) |
| fig1o-trial-end-ramp | no | metadata_insufficient (as above) |
| fig1-ramp-contralateral-only | no | metadata_insufficient (as above); 5 of 7 animals absent |
| figs1-running-speed | yes | |
| fig2a-success-rate | yes | |
| fig2b-lateral-spread | yes | |
| fig2c-psychometric | no | data_missing: psychometric animals and wall distances absent |
| fig2d-turn-angle | no | data_missing (as above) |
| fig2f-whisking-params | no | data_missing: no whisker tracking |
| fig2i-ft-rt | no | data_missing: no touch times; animals 573 and 480 absent |
| fig2jk-glm-mixed-tuning | no | data_missing: animal 579 and whisker and gait variables absent |
| fig2-epoch2-at-first-touch | no | data_missing: no touch times |
| fig2l-running-vs-sitting | no | data_missing: animal 480 absent |
| fig2m-dimensionality-collapse | no | metadata_insufficient: spike clock |
| figs12-tuning-lost-during-decision | no | data_missing: GLM variables absent |
| fig3-wall-kinematics | no | data_missing: no wall positions or touch times |
| fig3c-touch-statistics | no | data_missing: no touch times |
| fig3ijkl-whisking-across-epochs | no | data_missing: no whisker tracking |
| fig3-not-sensory-drive | no | data_missing: depends on touch statistics |
| fig4-ddm-fit | yes | |
| fig4b-ddm-matches-ramp | no | metadata_insufficient: the ramp onset needs the spike clock |
| fig5e-slope-across-animals | no | metadata_insufficient: spike clock; 5 of 7 animals absent |
| fig4f-varce-example-unit | no | metadata_insufficient: spike clock |
| fig4g-varce-population | no | metadata_insufficient: spike clock |
| fig4j-corce | no | metadata_insufficient: spike clock |
| fig4k-streak-index | no | metadata_insufficient: spike clock |
| fig5b-epochs-in-subpopulations | no | metadata_insufficient: spike clock |
| fig5cd-other-animals | no | metadata_insufficient: spike clock; 5 of 7 animals absent |
| fig5e-principal-vs-neighbor | no | metadata_insufficient: spike clock; 5 of 7 animals absent |


In [ ]:
import time
import warnings
from multiprocessing import Pool

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import barrel_vr as bv

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

started = time.time()
urls = bv.asset_urls()
S = {}
for s, u in urls.items():
    f = bv.open_session(u)
    tr = bv.load_trials(f)
    beh = bv.load_behaviour(f)
    seg = bv.trial_segments(beh)
    S[s] = dict(file=f, trials=tr, beh=beh, seg=seg, units=bv.load_units(f), spikes=bv.all_spikes(f),
                samples=bv.trial_table_from_samples(beh, seg))
print(f"dandiset {bv.DANDISET_ID} version {bv.DANDISET_VERSION}: sessions {list(S)}, "
      f"loaded in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output. First the session metadata and the
contents of each file.

In [ ]:
for s, d in S.items():
    f = d["file"]
    g = f["general/subject"]
    meta = {k: bv._str(g[k][()]) for k in ["subject_id", "age", "date_of_birth", "sex", "genotype", "weight"]}
    start = bv._str(f["session_start_time"][()])
    dob = pd.Timestamp(meta["date_of_birth"]).tz_localize(None).normalize()
    ses = pd.Timestamp(start).tz_localize(None).normalize()
    print(f"session {s}: start {start[:10]}, {meta}; days from birth to session: {(ses - dob).days}")
    print("  acquisition:", {k: (f['acquisition'][k]['data'].shape[0], bv._str(f['acquisition'][k]['data'].attrs['unit']))
                            for k in f["acquisition"]})
    print("  units table columns:", [k for k in f["units"].keys()])
print("Table S1: 489 female, P68, session 7/8/2022; 503 female, P89, session 9/2/2022.")

The sessions are 489 (2022-07-08) and 503 (2022-09-02), as in Table S1. The file of 489 gives its age as P63D,
but its date of birth and session date are 68 days apart, which is the P68 of Table S1. For 503 the file says P89D
and Table S1 P89; the dates are 88 days apart. Both files give a weight of 0.02 kg. Every acquisition series has a
unit; `trial` (unit "n") is the trial number of each sample and `ts_local` the time since that trial started.

Next, the trials table against the sample series.

In [ ]:
for s, d in S.items():
    tr = d["trials"]
    print(f"session {s}: {len(tr)} rows in the trials table; by label:", tr.label.value_counts().to_dict())
    print(f"  intact rows (before the first repeated start time): {tr.intact.sum()};",
          "by label:", tr[tr.intact].label.value_counts().to_dict())
    print(f"  rows with stop before start: {tr.negative.sum()} (all OL: {tr[tr.negative].label.str.startswith('OL').all()}),"
          f" rows {tr.index[tr.negative].min()} to {tr.index[tr.negative].max()}")
    it = tr[tr.intact]
    gap = it.start_time.values[1:] - it.start_time.values[:-1]
    neg = np.flatnonzero(it.negative.values[:-1])
    print(f"  for those rows, next start minus start: {np.round(np.unique(np.round(gap[neg], 3)), 3)} s")
    samp = d["samples"]
    j = tr.index[tr.negative][0]
    print(f"  example row {j}: table start {tr.start_time[j]:.3f}, stop {tr.stop_time[j]:.3f}; its samples run "
          f"{samp.t_first[j]:.3f} to {samp.t_last[j]:.3f} on the timestamps, {samp.t_local_end[j]:.3f} s on ts_local; "
          f"the straight row before it starts at {tr.start_time[j - 1]:.3f}")
    ok = it.index[~it.negative]
    print(f"  other intact rows: table start equals first sample time in {np.mean(np.isclose(samp.t_first[ok], it.start_time[ok], atol=0.003)):.3f}"
          f" of rows; OL durations {it[it.label.str.startswith('OL') & ~it.negative].duration.describe()[['min', '50%', 'max']].round(3).to_dict()}")

In both sessions, a run of OL rows (62 in 489, 63 in 503) has a stop time earlier than its start time. The samples
of these rows carry timestamps that restart at the preceding straight row's start, while `ts_local` runs from 0 to
2.0 s and the next row starts 2.0 s after the row's start. The start times are therefore taken as right, the
timestamps and stop times of those rows as wrong, and time within a trial is taken from `ts_local`. All other intact
rows start at their first sample's timestamp.

In 489, the trials table from row 1000 on (611 rows) repeats one start time, stop time and label (`OL_L_1.dat`).
The sample series below shows that these are real trials with their own times.

In [ ]:
d = S["489"]
tr, samp, beh, seg = d["trials"], d["samples"], d["beh"], d["seg"]
bad = tr.index[~tr.intact]
print(f"489: rows {bad.min()} to {bad.max()} ({len(bad)}) have start {tr.start_time[bad].unique()}, stop {tr.stop_time[bad].unique()},"
      f" label {list(tr.trial_type[bad].unique())}")
b = samp.loc[bad]
short = b.t_local_end < 2.2
print(f"  from the trial series: times {b.t_first.min():.1f} to {b.t_last.max():.1f} s; {short.sum()} rows last 2.0 s "
      f"(ts_local end {b.t_local_end[short].min():.3f} to {b.t_local_end[short].max():.3f}), {(~short).sum()} last "
      f"{b.t_local_end[~short].min():.2f} to {b.t_local_end[~short].max():.2f} s; they alternate: "
      f"{np.all(np.diff(short.values.astype(int)) != 0)}")
print(f"  forward distance at the end of the long rows: median {b.d_for_end[~short].median():.1f} cm (straight runs of 100 cm)")
x1 = bv.lateral_at(beh, seg, list(b.index[short]), 1.0)
print(f"  lateral displacement at 1 s in the 2-s rows: {np.sum(x1 > 0)} to the left, {np.sum(x1 < 0)} to the right,"
      f" median {np.nanmedian(x1):.2f} cm")
it = tr[tr.intact & tr.label.str.startswith("OL")]
for lab in ["OL_L", "OL_R"]:
    r = list(it.index[it.label == lab])
    print(f"  intact {lab} rows: lateral displacement at 1 s median {np.nanmedian(bv.lateral_at(beh, seg, r, 1.0)):.2f} cm")

The 611 repeated rows of 489 are 305 two-second turns alternating with 306 straight runs that end at 100 cm, over
3683 to 6218 s, while the table labels all of them OL_L at one time. Of the 305 turns, 256 go to the right and 49
to the left at 1 s, so the label is wrong for most of them and the turn type of each cannot be read from the file.
These rows are left out of every analysis; the session's labelled trials are its first 1000 rows (500 straight,
249 OL_L, 251 OL_R). This is the 1000 trials of Table S1.

Next, the ranges and units of the behavioural series.

In [ ]:
for s, d in S.items():
    tr, beh, seg, samp = d["trials"], d["beh"], d["seg"], d["samples"]
    dt = np.diff(beh.ts.values)
    print(f"session {s}: {len(beh)} samples; timestamp step {np.median(dt):.4f} s ({1 / np.median(dt):.0f} Hz, SI: 500 Hz);"
          f" steps that go back in time: {(dt < 0).sum()}")
    for c in ["d_for", "d_lat", "v_for", "v_lat"]:
        v = beh[c].values
        print(f"  {c}: {np.nanmin(v):.2f} to {np.nanmax(v):.2f}, median {np.nanmedian(v):.2f}, NaN {np.isnan(v).mean():.4f}")
    it = tr[tr.intact]
    cl = it.index[it.label == "CL"]
    print(f"  straight rows: forward distance at the end median {samp.d_for_end[cl].median():.1f} cm "
          f"(5th to 95th percentile {samp.d_for_end[cl].quantile(.05):.1f} to {samp.d_for_end[cl].quantile(.95):.1f});"
          f" duration median {samp.t_local_end[cl].median():.2f} s")
    # forward distance against the integral of forward speed, per straight row
    ratio = []
    for r in cl[samp.d_for_end[cl].values > 50]:
        idx = seg[r]
        t = beh.t_local.values[idx]
        v = np.nan_to_num(beh.v_for.values[idx])
        ratio.append(np.trapezoid(v, t) / max(beh.d_for.values[idx][-1], 1e-9))
    print(f"  {np.sum(samp.d_for_end[cl] <= 50)} straight rows end before 50 cm. Over the others, integral of v_for / forward distance: median {np.median(ratio):.3f}"
          f" (5th to 95th percentile {np.percentile(ratio, 5):.3f} to {np.percentile(ratio, 95):.3f})")

The ball tracking is sampled at 500 Hz, as the SI states. Forward distance restarts at each trial and the straight
rows end at 100 cm (the "100cm" of their labels), so distances are in cm. Over the straight rows that reach 50 cm,
the integral of `speed_for_smooth` over the row equals its forward distance (median ratio 1.00), so speeds are in
cm/s on the same clock. In 503, 169 straight rows end before 50 cm (5 in 489), and its straight rows last a median
10.00 s against 4.82 s in 489. Lateral distance also restarts at 0 at each trial.

Do the behavioural samples share the clock of the trials table? Below, the trial type (OL_L against OL_R) is
decoded from lateral position at single times around the stored turn starts, and from all samples after the start
with the true labels and with labels shifted by one OL trial.

In [ ]:
rng = np.random.default_rng(1)
for s, d in S.items():
    tr, beh, seg = d["trials"], d["beh"], d["seg"]
    ol = tr[tr.intact & tr.label.str.startswith("OL")]
    y = (ol.label == "OL_L").values.astype(int)
    rows = list(ol.index)
    grid = np.arange(-0.5, 1.0001, 0.1)
    X = np.array([[bv._on_grid(beh, seg[r], "d_lat", np.array([t]))[0] if t >= 0 else
                   bv._on_grid(beh, seg[r - 1], "d_lat", np.array([t + beh.t_local.values[seg[r - 1]][-1]]))[0]
                   for t in grid] for r in rows])
    X = np.nan_to_num(X)
    acc = {f"{t:+.1f}": bv.decode_accuracy(X[:, [k]], y, rng) for k, t in enumerate(grid)}
    shifted = bv.decode_accuracy(X[:, grid > 0], np.roll(y, 1), rng)
    true = bv.decode_accuracy(X[:, grid > 0], y, rng)
    print(f"session {s}: OL_L against OL_R from lateral position (cm) in single 0.1 s samples, accuracy by time from the turn start:")
    print("  ", {k: round(v, 2) for k, v in acc.items()})
    print(f"   from all samples after the start: true labels {true:.3f}; labels shifted by one OL trial {shifted:.3f};"
          f" chance {max(y.mean(), 1 - y.mean()):.3f}")

The trial type cannot be decoded before or at the stored start (accuracy 0.42 to 0.56 from -0.5 to 0.2 s) and
becomes decodable from 0.3 to 0.4 s on (0.93 and 0.98 at 1 s); with the labels shifted by one trial accuracy falls
to chance. Lateral position therefore shares the clock of the trials table, and OL_L trials turn to the positive
(left) side, as the labels say.

Next, the units table.

In [ ]:
for s, d in S.items():
    u = d["units"]
    fs = u.duration < bv.FS_DURATION_MS
    print(f"session {s}: {len(u)} units; layers {u.layer.value_counts().reindex(['23', '4', '5A', '5B', '6', 'WM']).fillna(0).astype(int).to_dict()}")
    print(f"  unit_type {u.unit_type.value_counts().to_dict()}; FS by duration < 0.4 ms: {fs.sum()}, RS: {(~fs).sum()};"
          f" unit_type Inhibitory and FS agree for {np.mean((u.unit_type == 'Inhibitory') == fs):.3f} of units")
    print(f"  duration range {u.duration.min():.2f} to {u.duration.max():.2f} ms; depth {u.depth.min()} to {u.depth.max()} um;"
          f" firing rate {u.firing_rate.min():.2f} to {u.firing_rate.max():.2f} Hz (mean {u.firing_rate.mean():.2f})")
    print(f"  quality (SI thresholds): amplitude cutoff <= 0.1 for {np.mean(u.amplitude_cutoff <= 0.1):.2f}, "
          f"ISI violations <= 0.5 for {np.mean(u.isi_viol <= 0.5):.2f}, presence ratio >= 0.9 for {np.mean(u.presence_ratio >= 0.9):.2f}")
    sp = d["spikes"]
    print(f"  spikes {len(sp)}, {sp.min():.2f} to {sp.max():.2f} s; trials table {d['trials'].start_time.min():.2f} to "
          f"{d['trials'].stop_time.max():.2f} s; behaviour samples to {np.nanmax(d['samples'].t_last):.2f} s")
u = S["489"]["units"]
print("\nUnit 101 of 489 (489101 in Fig. 4F and S10, 'RS, layer 5A'; Fig. S5E 'RS, layer 5B'):")
print(u[u.id == 101][["id", "layer", "unit_type", "duration", "depth", "firing_rate"]].to_string(index=False))
print("highest-rate unit of 489:", u.loc[u.firing_rate.idxmax(), ["id", "layer", "unit_type", "firing_rate"]].to_dict())

All units pass the SI's quality thresholds. `unit_type` (Excitatory, Inhibitory) is the 0.4 ms waveform duration
rule of the SI: Inhibitory equals duration below 0.4 ms for 61 of 61 units of 489 and 42 of 43 of 503. 489 has 61
units (Table S1: 65), with 7 in L2/3 and 12 in L6 where Fig. 5B gives 10 and 13, and 30 FS and 31 RS where Fig. 5B
gives 43 and 22. 503 has 43 units (Table S1: 48), three of them labelled WM, a layer the paper does not use. Unit 101
of 489, the paper's 489101, is labelled Inhibitory, L5A, waveform duration 0.2 ms (FS by the SI's rule), while the
paper calls it RS; it is the session's highest-rate unit (37.6 Hz), as the paper says of 489101. The spikes of 489
run to 6255 s, past the end of its trials table (3683 s) and its behaviour (6218 s); those of 503 run to 7326 s,
past the end of its behaviour (7297 s).

The spikes have to share the trial clock for any neural claim. The paper describes, for 489, a sharp peak at
0.026 s, a suppression to about a third of the baseline rate and a ramp in OL_L trials only, in all units. Below,
the population rate around the stored onsets, and the decoding of trial type from per-unit spike counts.

In [ ]:
edges = np.arange(-1.0, 2.0001, 0.005)
centres = edges[:-1] + 0.0025
psth = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, (s, d) in zip(axes, S.items()):
    tr, sp, nu = d["trials"], d["spikes"], len(d["units"])
    for lab, col in [("OL_L", "tab:red"), ("OL_R", "tab:blue")]:
        on = tr.start_time.values[(tr.intact & (tr.label == lab)).values]
        r = bv.population_psth(sp, nu, on, edges)
        psth[(s, lab)] = r
        ax.plot(centres, r, color=col, lw=0.7, label=f"{lab} ({len(on)})")
        e1 = r[(centres > -1) & (centres < 0)]
        print(f"{s} {lab} at the stored onsets: epoch 1 (-1 to 0 s) {e1.mean():.2f} Hz (SD over 5 ms bins {e1.std():.2f});"
              f" max 0 to 0.065 s {r[(centres > 0) & (centres < 0.065)].max():.2f}; min 0.065 to 0.14 s "
              f"{r[(centres > 0.065) & (centres < 0.14)].min():.2f}; max 0.12 to 0.3 s {r[(centres > 0.12) & (centres < 0.3)].max():.2f};"
              f" range over -1 to 2 s {r.min():.2f} to {r.max():.2f}")
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"{s}: population rate at the stored OL onsets")
    ax.set_xlabel("time from stored onset (s)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Hz per unit (5 ms bins)")
plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(2)
for s, d in S.items():
    tr, sp, u = d["trials"], d["spikes"], d["units"]
    f = d["file"]
    idx = f["units/spike_times_index"][:]
    st = f["units/spike_times"][:]
    per_unit = np.split(st, idx[:-1])
    ol = tr[tr.intact & tr.label.str.startswith("OL")]
    y = (ol.label == "OL_L").values.astype(int)
    res = {}
    for name, (a, b) in {"-0.5 to 0 s": (-0.5, 0), "0 to 0.5 s": (0, 0.5), "0.5 to 1.8 s": (0.5, 1.8)}.items():
        X = np.array([[np.searchsorted(x, t0 + b) - np.searchsorted(x, t0 + a) for x in per_unit] for t0 in ol.start_time.values], float)
        acc = bv.decode_accuracy(X, y, rng)
        null = [bv.decode_accuracy(X, rng.permutation(y), rng) for _ in range(20)]
        res[name] = f"{acc:.3f} (shuffled labels {np.mean(null):.3f}, 95th percentile {np.percentile(null, 95):.3f})"
    print(f"session {s}: OL_L against OL_R from per-unit spike counts at the stored onsets: {res}")

At the stored onsets there is no trial-locked response. The population rate stays within 5.8 to 10.8 Hz over
-1 to 2 s, with a baseline SD of 0.41 to 0.50 Hz over 5 ms bins. The lowest bin of the suppression window (0.065
to 0.14 s) is 6.43 to 8.49 Hz, at least 86% of the baseline (489 OL_L: 6.43 against 7.48 Hz), where the paper has
the rate of 489 fall from 2 to 0.7 Hz, and the largest bin of the ramp window is no higher than the largest bins
elsewhere in -1 to 2 s (489 OL_L: 8.01 Hz, range 6.08 to 8.69). Trial
type cannot be decoded from the units' spike counts in any window (0.46 to 0.52, against shuffled-label 95th
percentiles of 0.52 to 0.55), although the animal's lateral position decodes it at 0.92 to 0.96.

The spikes may be offset from the trial clock. The tables below search for the offset two ways: the lag of the
largest onset-triggered deviation of the population rate, per window of about 40 onsets, and the lag of the largest
correlation between the population rate and running speed, per 400 s window. Neither uses the shape of the paper's
response.

In [ ]:
lags = np.arange(-20, 60, 0.005)
rows = []
for s, d in S.items():
    tr, sp = d["trials"], d["spikes"]
    on = tr.start_time.values[(tr.intact & tr.label.str.startswith("OL")).values]
    for k, chunk in enumerate(np.array_split(on, len(on) // 40)):
        c = bv.lag_scan(sp, chunk, lags)
        lag, val, z = bv.best_lag(lags, c)
        rows.append(dict(session=s, window=k, t_from=chunk[0], t_to=chunk[-1], n_onsets=len(chunk),
                         best_lag=lag, value_Hz=val, z=z))
lag_tab = pd.DataFrame(rows)
print("Lag (s) of the largest onset-triggered deviation of the summed population rate (50 ms smoothing), per window of about"
      " 40 OL onsets, searched over -20 to 60 s:")
print(lag_tab.round(3).to_string(index=False))

In [ ]:
lags_v = np.arange(0, 40, 0.05)
rows = []
for s, d in S.items():
    tr, beh, seg, sp = d["trials"], d["beh"], d["seg"], d["spikes"]
    t_beh = bv.behaviour_time(beh, tr, seg)
    speed = np.hypot(beh.v_for.values, beh.v_lat.values)
    t_end = np.nanmax(t_beh)
    for t0 in np.arange(0, t_end - 400, 400):
        r = bv.speed_lag_scan(sp, t_beh, speed, t0, t0 + 400, lags_v)
        k = np.argmax(np.abs(r))
        rows.append(dict(session=s, window_from=t0, best_lag=lags_v[k], r=r[k], r_at_0=r[0], median_abs_r=np.median(np.abs(r))))
speed_tab = pd.DataFrame(rows)
print("Lag (s) of the largest correlation between population spike count and running speed, per 400 s window of the trial clock:")
print(speed_tab.round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3.2))
for s, mk in [("489", "o"), ("503", "s")]:
    a = lag_tab[lag_tab.session == s]
    b = speed_tab[speed_tab.session == s]
    ax.plot((a.t_from + a.t_to) / 2, a.best_lag, mk + "-", label=f"{s}, OL onsets")
    ax.plot(b.window_from + 200, b.best_lag, mk + "--", label=f"{s}, running speed")
ax.set_xlabel("time on the trial clock (s)")
ax.set_ylabel("best lag of spikes (s)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"plotted lags range {min(lag_tab.best_lag.min(), speed_tab.best_lag.min()):.2f} to "
      f"{max(lag_tab.best_lag.max(), speed_tab.best_lag.max()):.2f} s; all points within the axes")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
edges25 = np.arange(-1.0, 2.0001, 0.025)
c25 = edges25[:-1] + 0.0125
for ax, (s, d) in zip(axes, S.items()):
    tr, sp, nu = d["trials"], d["spikes"], len(d["units"])
    first = lag_tab[(lag_tab.session == s) & (lag_tab.window == 0)].iloc[0]
    for lab, col in [("OL_L", "tab:red"), ("OL_R", "tab:blue")]:
        on = tr.start_time.values[(tr.intact & (tr.label == lab)).values]
        on = on[on <= first.t_to]
        r = bv.population_psth(sp, nu, on + first.best_lag, edges25)
        ax.plot(c25, r, color=col, label=f"{lab} ({len(on)})")
        print(f"{s} {lab}, onsets of window 0 shifted by {first.best_lag:.3f} s: rate {r.min():.2f} to {r.max():.2f} Hz;"
              f" -1 to 0 s mean {r[c25 < 0].mean():.2f}, 0 to 0.3 s mean {r[(c25 > 0) & (c25 < 0.3)].mean():.2f},"
              f" 0.3 to 1.8 s mean {r[(c25 > 0.3) & (c25 < 1.8)].mean():.2f}")
    ax.set_title(f"{s}: window 0 onsets + {first.best_lag:.2f} s")
    ax.set_xlabel("time from shifted onset (s)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Hz per unit (25 ms bins)")
plt.tight_layout()
plt.show()

**The spikes cannot be placed on the trial clock.** The best lag of the onset-triggered rate grows within each
session (489: 4.4, 6.8, 7.5, 7.4, 9.6, 10.9, 12.2 s over the first seven windows; 503: 5.6, 11.8, 14.2 s), and
later windows jump (43.3 s in 489, 52.5 s in 503) at deviations of z 3.1 to 9.9, which a search over 16,000 lags
can reach by chance. The speed correlation gives a smooth drift in 503 (12.9 s rising to 19.8 s over the first
5200 s), which does not agree with the onset estimate in the same windows (5.6 and 11.8 s), and in 489 its
correlations are weak (|r| at most 0.17) and change sign. At the best lag of the first window the trial-locked rate
shows broad shifts of the whole window (503 OL_L: 4.18, 3.93 and 4.30 Hz in the three windows, 489 OL_R 3.77, 2.98,
6.24 Hz), not the paper's peak, suppression and ramp. The files hold no synchronisation signal and do not
say how spike times relate to the trial times. An alignment fitted to the neural response would set the very
timings the claims test (the 0.026 s peak, the 0.12 s ramp onset), so none is adopted, and every claim that needs
spikes on the trial clock is blocked as `metadata_insufficient`. The behaviour is on the trial clock and its claims
are tested.

The behaviour claims use the intact rows: 489 rows 0 to 999 (249 OL_L, 251 OL_R) and all 1027 rows of 503
(230 OL_L, 283 OL_R).

## table-s1-recordings

> The recordings comprise 13 ephys sessions and 6 behaviour-only sessions, one per animal, with the numbers of
> accepted single units and of trials per animal listed in Table S1, and with the units and OL turn trials of the
> example sessions given in Fig. 1J-K and Fig. S4.

Criterion: unit counts and the OL_L plus OL_R counts must match exactly, and at least 17 of 19 total trial counts
within 10%; partially reproduced when unit counts match for at least 10 of 13 ephys animals. The paper's numbers
below are the claims file's.

In [ ]:
# Table S1 and the legends of Fig. 1J-K and Fig. S4, as the claims file gives them
paper_units = {"453": 56, "454": 40, "456": 39, "480": 42, "482": 38, "488": 29, "489": 65, "503": 48, "516": 33,
               "517": 42, "528": 89, "530": 113, "579": 34}
paper_trials = {"453": 2563, "454": 2581, "456": 2262, "474": 131, "475": 55, "480": 758, "482": 1518, "488": 939,
                "489": 1000, "499": 1123, "503": 1027, "516": 1071, "517": 1287, "518": 1291, "528": 525, "530": 1654,
                "573": 401, "579": 55, "589": 50}
paper_ol = {"489": (266, 265), "503": (305, 304), "528": (95, 95), "530": (402, 403), "482": (350, 350),
            "480": (185, 190), "517": (270, 268)}
rows = []
for a in paper_trials:
    d = S.get(a)
    row = dict(animal=a, in_dataset=d is not None, units_paper=paper_units.get(a), trials_paper=paper_trials[a],
               OL_L_paper=paper_ol.get(a, (None, None))[0], OL_R_paper=paper_ol.get(a, (None, None))[1])
    if d is not None:
        tr = d["trials"]
        row.update(units=len(d["units"]), trials_rows=len(tr), trials_intact=int(tr.intact.sum()),
                   OL_L_rows=int((tr.label == "OL_L").sum()), OL_R_rows=int((tr.label == "OL_R").sum()),
                   OL_L_intact=int((tr.intact & (tr.label == "OL_L")).sum()), OL_R_intact=int((tr.intact & (tr.label == "OL_R")).sum()))
        if a == "489":
            samp = d["samples"]
            bad = tr.index[~tr.intact]
            row["OL_rows_block2_by_duration"] = int((samp.loc[bad].t_local_end < 2.2).sum())
    rows.append(row)
t1 = pd.DataFrame(rows)
print(t1.to_string(index=False))
present = t1[t1.in_dataset]
print(f"\nanimals in the dataset: {len(present)} of {len(t1)} (ephys {len(present)} of {len(paper_units)})")
for _, r in present.iterrows():
    print(f"{r.animal}: units {r.units} vs {r.units_paper} (match {r.units == r.units_paper}); trials {r.trials_rows} rows,"
          f" {r.trials_intact} intact, vs {r.trials_paper} (within 10%: rows {abs(r.trials_rows - r.trials_paper) <= 0.1 * r.trials_paper},"
          f" intact {abs(r.trials_intact - r.trials_paper) <= 0.1 * r.trials_paper}); OL_L + OL_R intact "
          f"{r.OL_L_intact} + {r.OL_R_intact} = {r.OL_L_intact + r.OL_R_intact} vs {r.OL_L_paper} + {r.OL_R_paper} = "
          f"{r.OL_L_paper + r.OL_R_paper}")
u = S["489"]["units"]
fs = u.duration < bv.FS_DURATION_MS
print("\n489 units per layer against Fig. 5B (L2/3 10, L4 13, L5A 13, L5B 16, L6 13; FS 43, RS 22):",
      u.layer.value_counts().reindex(["23", "4", "5A", "5B", "6"]).to_dict(), f"FS {fs.sum()}, RS {(~fs).sum()};"
      f" unit_type Inhibitory {np.sum(u.unit_type == 'Inhibitory')}, Excitatory {np.sum(u.unit_type == 'Excitatory')}")

**Result: not reproduced.** The dataset holds 2 of the 19 animals (2 of 13 ephys sessions), so 17 trial counts and
11 unit counts cannot be compared. Of the two present, neither unit count matches (489: 61 against 65; 503: 43
against 48), and neither OL count does (489: 249 + 251 = 500 intact against 266 + 265 = 531; 503: 230 + 283 = 513
against 305 + 304 = 609). The trial counts match: 503 has 1027 rows (1027) and 489 has 1000 intact rows (1000),
though its table has 1611 rows. The 489 layer counts differ from Fig. 5B in L2/3 (7 against 10) and L6 (12 against
13), and its FS and RS counts (30 and 31) differ from Fig. 5B (43 and 22).

## figs1-running-speed

> Mice run at fast speeds, 21 ± 4 cm/s (mean ± SD).

Criterion: absolute speed from forward and lateral velocity; readings (a) 503, mean over samples; (b) 503,
samples of running trials (mean trial speed above 10 cm/s); (c) the mean of the per-animal mean running speeds over
19 animals, with each animal compared with Table S1 within 10%; (d) habituation sessions if present. Reproduced
when the mean is within 4 cm/s of 21 under a reading.

In [ ]:
paper_speed = {"489": (92.6, 22.2), "503": (52.8, 16.1)}   # Table S1: run %, mean running speed (cm/s)
rows = []
for s, d in S.items():
    tr, beh, seg = d["trials"], d["beh"], d["seg"]
    for scope, rws in [("all rows", list(seg)), ("intact rows", list(tr.index[tr.intact]))]:
        samples, per_trial = bv.session_speed(beh, seg, rws)
        run = per_trial[per_trial > bv.RUN_THRESHOLD]
        run_samples, _ = bv.session_speed(beh, seg, list(run.index))
        rows.append(dict(session=s, scope=scope, trials=len(per_trial),
                         a_mean_samples=np.nanmean(samples), a_sd_samples=np.nanstd(samples),
                         b_mean_samples_running=np.nanmean(run_samples), b_sd=np.nanstd(run_samples),
                         run_pct=100 * len(run) / len(per_trial), mean_running_trial_speed=run.mean(),
                         run_pct_paper=paper_speed[s][0], mean_speed_paper=paper_speed[s][1]))
sp_tab = pd.DataFrame(rows)
print(sp_tab.round(2).to_string(index=False))
main = sp_tab[sp_tab.scope == "all rows"].set_index("session")
print(f"\n(a) 503, mean absolute speed over samples: {main.a_mean_samples['503']:.2f} cm/s (SD {main.a_sd_samples['503']:.2f})")
print(f"(b) 503, samples of running trials (mean trial speed > 10 cm/s): {main.b_mean_samples_running['503']:.2f} cm/s (SD {main.b_sd['503']:.2f})")
c = main.mean_running_trial_speed
print(f"(c) mean of the per-animal mean running speeds, the 2 animals in the dataset: {c.mean():.2f} cm/s ({c.round(2).to_dict()});"
      " the other 17 animals are not in the dataset")
for s in main.index:
    print(f"    {s}: mean running speed {c[s]:.2f} vs Table S1 {paper_speed[s][1]} (within 10%: {abs(c[s] - paper_speed[s][1]) <= 0.1 * paper_speed[s][1]});"
          f" run % {main.run_pct[s]:.1f} vs {paper_speed[s][0]}")
print("(d) habituation sessions: the dataset holds one session per animal, the terminal one")
fig, ax = plt.subplots(figsize=(5, 3))
samples, _ = bv.session_speed(S["503"]["beh"], S["503"]["seg"])
ax.hist(samples, bins=np.arange(0, 60, 1), color="0.4")
ax.set_xlabel("absolute speed, 503 (cm/s)")
ax.set_ylabel("samples")
plt.tight_layout()
plt.show()
print(f"histogram range 0 to 60 cm/s; {np.sum(samples >= 60)} of {len(samples)} samples above it, max {np.nanmax(samples):.1f}")

**Result: not reproduced.** Animal 503 runs at a mean of 5.93 cm/s over all samples (reading a) and 12.95 cm/s
over the samples of its running trials (reading b), both outside 17 to 25 cm/s. Reading (c) can use only 2 of the 19
animals; their mean running speed is 16.56 cm/s, also outside. 489 alone (19.98 cm/s) would be inside. The dataset
holds no habituation sessions (reading d). Against Table S1, 489 matches (19.98 against 22.2 cm/s; 91.7% running
trials against 92.6%), but 503 does not (13.14 against 16.1 cm/s; 27.7% against 52.8%).

## fig2a-success-rate

> In the unrewarded task, with a correct trial defined as ±1 cm lateral clearance at t = 1 s, animal 503 succeeds
> in 95% of 414 randomized turn trials (14 failed trajectories).

Criterion: animal 503, OL trials, lateral position at t = 1 s; zero (e) at t = 0 or (f) at the start of the
straight section; trials (a) all OL or (b) the first 414; clearance (c) at least 1 cm in the turn direction or (d)
at least 1 cm either way. Reproduced when the success rate is within 10% of 0.95 under a reading. A variant, not in
the criterion, uses only OL trials run above 10 cm/s (Table S1's running criterion), since the paper does not say
which 414 trials it used.

In [ ]:
d = S["503"]
tr, beh, seg = d["trials"], d["beh"], d["seg"]
ol = tr[tr.intact & tr.label.str.startswith("OL")]
rows_all = list(ol.index)
sign = np.where(ol.label == "OL_L", 1, -1)
x1 = bv.lateral_at(beh, seg, rows_all, 1.0)          # lateral displacement at t = 1 s from t = 0 (reading e)
_, per_trial = bv.session_speed(beh, seg, rows_all)
st, cum = bv.lateral_traces(beh, seg, rows_all, np.array([1.0]))
x1_cum = cum[:, 0]                                     # from the start of the straight section (reading f)
res = []
for zero, x in [("e: from t = 0", x1), ("f: from the CL start", x1_cum)]:
    for clear, ok in [("c: >= 1 cm in turn direction", sign * x >= 1), ("d: |displacement| >= 1 cm", np.abs(x) >= 1)]:
        for trials, m in [("a: all OL trials", np.ones(len(x), bool)), ("b: first 414 OL trials", np.arange(len(x)) < 414),
                          ("variant: running OL trials", (per_trial.values > bv.RUN_THRESHOLD))]:
            res.append(dict(zero=zero, clearance=clear, trials=trials, n=m.sum(), success=ok[m].mean(), failed=int((~ok[m]).sum())))
succ = pd.DataFrame(res)
print(f"503: {len(ol)} OL trials in the dataset ({(ol.label == 'OL_L').sum()} OL_L, {(ol.label == 'OL_R').sum()} OL_R); "
      f"no displacement at 1 s for {np.isnan(x1).sum()}")
print(succ.round(3).to_string(index=False))
print(f"\nwithin 10% of 0.95 (0.855 to 1.045): {succ[succ.trials != 'variant: running OL trials'].success.between(0.855, 1.045).sum()}"
      f" of {np.sum(succ.trials != 'variant: running OL trials')} criterion readings")
for lab in ["OL_L", "OL_R"]:
    m = (ol.label == lab).values
    print(f"{lab}: displacement at 1 s in the turn direction, median {np.nanmedian(sign[m] * x1[m]):.2f} cm, "
          f"5th to 95th percentile {np.nanpercentile(sign[m] * x1[m], 5):.2f} to {np.nanpercentile(sign[m] * x1[m], 95):.2f}")

**Result: not reproduced.** Under the eight readings of the criterion the success rate is 0.609 to 0.694 (154 to
165 failed trials), against 0.95 (0.855 to 1.045). The failures are trials with almost no lateral movement: the
5th percentile of displacement in the turn direction at 1 s is 0.00 cm for both turn types. Restricted to the 94 OL
trials run above 10 cm/s, a variant outside the criterion, the success rate is 0.979 to 1.000 (0 to 2 failures),
so the paper's 95% may come from a selection of running trials that the paper does not describe.

## fig2b-lateral-spread

> Lateral positions are tightly confined during the straight CL section (0.14 ± 0.55 cm, mean ± SD, at t = 0) and
> their distribution widens over time after the turn (t = 1 s and 3 s), about twice as wide at 3 s as at 2 s (SD
> 2.30 against 1.49).

Criterion: 503 (Fig. 2B) and 489 (Fig. S6B); lateral distance (a) as stored or (b) integrated from the start of
the straight section; the 2 s and 3 s SDs read as (c) 503 at 3 s and 489 at 2 s, (d) both 503, (e) both 489.
Reproduced when the t = 0 values and the widening hold and the 2 s and 3 s SDs match under one reading; partially
when only the t = 0 values and the widening hold. Following the legends, 503 pools OL_L and OL_R (Fig. 2B, "414
trials randomized between OL_L and OL_R") and 489 uses OL_L (Fig. S6A-B); per-direction values for 503 are shown
as well.

In [ ]:
grid = np.array([0.0, 1.0, 2.0, 3.0])
rows = []
traces = {}
for s, labs in [("503", ["OL_L", "OL_R"]), ("489", ["OL_L"])]:
    d = S[s]
    tr, beh, seg = d["trials"], d["beh"], d["seg"]
    for scope, ls in [("pooled as in the legend", labs), ("OL_L", ["OL_L"]), ("OL_R", ["OL_R"])]:
        if s == "489" and scope != "pooled as in the legend":
            continue
        rws = list(tr.index[tr.intact & tr.label.isin(ls)])
        st, cum = bv.lateral_traces(beh, seg, rws, grid)
        traces[(s, scope)] = (st, cum)
        for rd, a in [("a: stored", st), ("b: from the CL start", cum)]:
            rows.append(dict(session=s, trials=scope, n=len(rws), reading=rd,
                             **{f"mean_t{t:g}": np.nanmean(a[:, k]) for k, t in enumerate(grid)},
                             **{f"sd_t{t:g}": np.nanstd(a[:, k]) for k, t in enumerate(grid)}))
spread = pd.DataFrame(rows)
print("Lateral distance (cm) at t = 0, 1, 2 and 3 s from the turn start. Fig. 2B: 503, OL_L and OL_R pooled;"
      " Fig. S6B: 489, OL_L.")
print(spread.round(2).to_string(index=False))
print("\nPaper: t = 0 mean 0.14, SD 0.55 (tolerance: mean within 0.55, SD within 0.055); SD 2.30 at 3 s, 1.49 at 2 s.")
for rd in ["a: stored", "b: from the CL start"]:
    p = spread[(spread.reading == rd) & (spread.trials == "pooled as in the legend")].set_index("session")
    print(f"{rd}: 503 t = 0 mean {p.mean_t0['503']:.2f}, SD {p.sd_t0['503']:.2f}; widening 0 -> 1 -> 3 s (503): "
          f"{p.sd_t0['503']:.2f} -> {p.sd_t1['503']:.2f} -> {p.sd_t3['503']:.2f};"
          f" (c) 503 SD at 3 s {p.sd_t3['503']:.2f}, 489 SD at 2 s {p.sd_t2['489']:.2f};"
          f" (d) 503 at 3 s {p.sd_t3['503']:.2f} and 2 s {p.sd_t2['503']:.2f}; (e) 489 at 3 s {p.sd_t3['489']:.2f} and 2 s {p.sd_t2['489']:.2f}")

**Result: not reproduced.** The stored lateral distance restarts at 0 at every trial, so at t = 0 it is 0.00 ± 0.00
cm by construction (reading a). Integrated from the start of the straight section (reading b), 503 is at 0.20 ±
0.84 cm at t = 0: the mean is within 0.55 cm of 0.14, but the SD is outside 0.55 ± 0.055. Under (b) the distribution
widens (SD 0.84, 2.90 and 7.20 cm at 0, 1 and 3 s for 503), but none of the 2 s and 3 s SDs matches 2.30 and 1.49
under (c), (d) or (e) (503: 6.64 and 7.20 cm; 489: 3.54 and 5.40 cm; under (a), 503 0.76 at 3 s and 489 3.85 at 2 s).
Pooling both turn directions widens the 503 distributions; per direction its 3 s SDs are 4.58 (OL_L) and 3.10
(OL_R) cm under (b).

## fig4-ddm-fit

> A classical drift-diffusion model fitted to the reaction-time distribution of animal 489 gives a nondecision time
> of 0.12 s, drift 2.91, bound 0.59 and noise 0.25 (BIC -664); a generalized DDM with collapsing bounds and leak fits
> better (BIC -705), with normalized drift K/B of 11.12.

Criterion: animal 489, OL_L trials ((e) the 266 of Fig. 4B or (f) the 520 of Fig. S7C-D; the dataset's labelled
OL_L trials are 249, and all are used); RT as in fig2i-ft-rt: the largest peak after t = 0 of (c) the run angle
derivative or (d) the lateral acceleration, in the turn direction, searched over (e) 0 to 1 s or (f) 0 to 1.8 s;
correct as in fig2a-success-rate; PyDDM; (a) sigma free or (b) sigma fixed at 0.25; GDDM with exponentially
collapsing bounds and a leaky integrator. Reproduced when NDT is within 10% of 0.12 s and the GDDM BIC is below
the DDM BIC under a reading; partially when only one holds. A maximum at the edge of the search window is not a
peak, and those trials are dropped (a standard choice, counted below). The first table also gives RT for both ephys
animals and both turn types, which fig2i-ft-rt asks for.

In [ ]:
rt_rows = []
RT = {}
for s, d in S.items():
    tr, beh, seg = d["trials"], d["beh"], d["seg"]
    for lab, sg in [("OL_L", 1), ("OL_R", -1)]:
        rws = list(tr.index[tr.intact & (tr.label == lab)])
        x1 = bv.lateral_at(beh, seg, rws, 1.0)
        for sig in ["angle", "acc"]:
            for w in [(0.0, 1.0), (0.0, 1.8)]:
                rt = np.array([bv.reaction_time(beh, seg[r], sg, sig, w) for r in rws])
                edge = (rt <= w[0] + bv.RT_DT) | (rt >= w[1] - bv.RT_DT) | np.isnan(rt)
                RT[(s, lab, sig, w)] = (rt, edge, x1, sg)
                ok = rt[~edge]
                rt_rows.append(dict(session=s, turn=lab, n=len(rws), signal=sig, window=f"{w[0]:g}-{w[1]:g} s",
                                    at_window_edge=int(edge.sum()), mean=ok.mean(), sd=ok.std(), median=np.median(ok), min=ok.min()))
rt_tab = pd.DataFrame(rt_rows)
print("Reaction time (s): time of the largest peak of the run angle derivative ('angle') or lateral acceleration ('acc')"
      " in the turn direction; maxima at the window edges are not peaks and are dropped. Paper (Fig. 2I, animal 573 or 480,"
      " not in the dataset): 0.38 +- 0.07 s.")
print(rt_tab.round(3).to_string(index=False))

In [ ]:
jobs = []
for sig in ["angle", "acc"]:
    for w in [(0.0, 1.0), (0.0, 1.8)]:
        rt, edge, x1, sg = RT[("489", "OL_L", sig, w)]
        for clear, correct in [("c", sg * x1 >= 1), ("d", np.abs(x1) >= 1)]:
            keep = ~edge & ~np.isnan(x1)
            for sigma_free in [True, False]:
                for model in ["DDM", "GDDM"]:
                    if model == "GDDM" and (sigma_free or w[1] > 1.0):
                        continue   # GDDM fits take 50 to 370 s each here; fitted on the sigma 0.25 readings of the 0 to 1 s window only
                    key = (sig, f"{w[0]:g}-{w[1]:g}", clear, "a: sigma free" if sigma_free else "b: sigma 0.25", model)
                    jobs.append((key, rt[keep], correct[keep], model, sigma_free, w[1]))
t_fit = time.time()
with Pool(4) as pool:
    fits = dict(pool.map(bv.fit_ddm, jobs))
print(f"{len(jobs)} fits in {time.time() - t_fit:.0f} s; trials per fit {len(jobs[0][1])} of "
      f"{len(RT[('489', 'OL_L', 'angle', (0.0, 1.0))][0])} OL_L (the rest have the maximum at a window edge or no displacement)")
ft = pd.DataFrame([dict(signal=k[0], window=k[1], clearance=k[2], sigma_reading=k[3], model=k[4], **v) for k, v in fits.items()])
ft["K_over_B"] = ft.K / ft.B
ft["correct_fraction"] = [jobs[i][2].mean() for i in range(len(jobs))]
print(ft.round(3).to_string(index=False))

In [ ]:
piv = ft.pivot_table(index=["signal", "window", "clearance", "sigma_reading"], columns="model", values=["NDT", "BIC", "K_over_B"])
piv.columns = [f"{a}_{b}" for a, b in piv.columns]
piv["NDT_DDM_within_10pct"] = (piv.NDT_DDM - 0.12).abs() <= 0.012
piv["GDDM_BIC_lower"] = piv.BIC_GDDM < piv.BIC_DDM
print(piv.round(3).to_string())
print(f"\nreadings with DDM NDT within 0.012 s of 0.12 s: {piv.NDT_DDM_within_10pct.sum()} of {len(piv)};"
      f" with GDDM BIC below DDM BIC: {piv.GDDM_BIC_lower.sum()} of {len(piv)}; both: {(piv.NDT_DDM_within_10pct & piv.GDDM_BIC_lower).sum()}")
dd = ft[ft.model == "DDM"]
print(f"DDM NDT range {dd.NDT.min():.3f} to {dd.NDT.max():.3f} s")
for col, ref, tol in [("K", 2.91, 0.291), ("B", 0.59, 0.059), ("sigma", 0.25, 0.025), ("BIC", -664, 66.4)]:
    print(f"  DDM {col}: {dd[col].min():.3f} to {dd[col].max():.3f}; within tolerance of {ref} in {np.sum((dd[col] - ref).abs() <= tol)} of {len(dd)} fits")
gd = ft[ft.model == "GDDM"]
print(f"  GDDM BIC {gd.BIC.min():.1f} to {gd.BIC.max():.1f} (paper -705, within 70.5 in {np.sum((gd.BIC + 705).abs() <= 70.5)} of {len(gd)});"
      f" K/B (c, GDDM) {gd.K_over_B.min():.2f} to {gd.K_over_B.max():.2f}, (d, DDM) {dd.K_over_B.min():.2f} to {dd.K_over_B.max():.2f};"
      f" paper 11.12 +- 1.11")
fig, ax = plt.subplots(figsize=(5, 3))
rt, edge, x1, sg = RT[("489", "OL_L", "angle", (0.0, 1.0))]
ax.hist(rt[~edge & (x1 >= 1)], bins=np.arange(0, 1.0001, 0.025), color="tab:red", alpha=0.7, label="correct")
ax.hist(rt[~edge & (x1 < 1)], bins=np.arange(0, 1.0001, 0.025), color="k", alpha=0.7, label="error")
ax.set_xlabel("RT, run angle derivative, 0 to 1 s (s)")
ax.set_ylabel("OL_L trials, 489")
ax.legend()
plt.tight_layout()
plt.show()
print("histogram covers 0 to 1 s, the whole search window")

**Result: partially reproduced.** The GDDM fits better than the DDM in each of the 4 readings where both were fitted
(BIC -201.6 to -76.8 against 42.7 to 82.8), but the DDM's nondecision time is 0.000 to 0.005 s in all 16 readings,
far from 0.12 s, so the two conditions never hold together. The other values are also far from the paper's: DDM
BIC 42.7 to 205.9 (paper -664), GDDM BIC -201.6 to -76.8 (paper -705), bound B 0.150 to 1.214 (0.59, in none of 16
fits) and K/B 2.50 to 3.80 (11.12). Drift K is within tolerance in 6 of 16 fits, but only where sigma is free, and
with sigma free the DDM's K, B and sigma trade off (only their ratios are identified). The GDDM leak sits at the
-10 edge of its range in all four fits. The near-zero NDT follows from the RT distribution: in 489 the RTs of OL_L
trials have a mean of 0.297 to 0.352 s and a minimum of 0.007 s, against the 0.38 ± 0.07 s of Fig. 2I,
which is from other animals. The GDDM was fitted only under sigma 0.25 and the 0 to 1 s window, because its fits
took minutes each while this notebook was prepared (not run here), too long for all 16 readings.

## Blocked claims

**Spike clock (`metadata_insufficient`).** fig1lm-staged-modulation, fig1n-epoch-rates, fig1jk-all-units,
fig1n-ramp-slope, fig1o-trial-end-ramp, fig1-ramp-contralateral-only, fig2m-dimensionality-collapse,
fig4b-ddm-matches-ramp, fig5e-slope-across-animals, fig4f-varce-example-unit, fig4g-varce-population, fig4j-corce,
fig4k-streak-index, fig5b-epochs-in-subpopulations, fig5cd-other-animals and fig5e-principal-vs-neighbor all
analyse spikes aligned to the turn onset. The data check shows that the spikes are not on the clock of the trials
table and that no alignment can be read from the files. The claims that compare animals also lack 5 of their 7
animals (only 489 and 503 are in the dataset).

**Data not in the dataset (`data_missing`).** fig2c-psychometric and fig2d-turn-angle need the psychometric animals
(474, 475, 499, 518) and the OL wall distance; fig2f-whisking-params, fig2i-ft-rt, fig2-epoch2-at-first-touch,
fig3-wall-kinematics, fig3c-touch-statistics, fig3ijkl-whisking-across-epochs and fig3-not-sensory-drive need
whisker tracking, touch times or wall positions, none of which are in the files, and animals 573 and 589;
fig2jk-glm-mixed-tuning and figs12-tuning-lost-during-decision need animal 579 and the whisker and gait regressors;
fig2l-running-vs-sitting is judged on animal 480. The paper names no other archive for these data.

## Summary

| Claim | Outcome |
|---|---|
| table-s1-recordings | not reproduced |
| figs1-running-speed | not reproduced |
| fig2a-success-rate | not reproduced |
| fig2b-lateral-spread | not reproduced |
| fig4-ddm-fit | partially reproduced |
| 16 claims on trial-aligned spikes | blocked: metadata_insufficient |
| 12 claims on whisker, touch, wall, gait or absent animals | blocked: data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

**How the analysis differs from the paper's.** Only animals 489 and 503 are available. In 489 only the first 1000
trials carry usable labels and times (249 OL_L and 251 OL_R, where the paper uses 266 and 265, or 520 OL_L). The
paper's trial selections (414 trials of 503, the 266 OL_L of 489) cannot be identified. Reaction-time maxima at the
edge of the search window are dropped. The GDDM was fitted only under the sigma 0.25 readings of the 0 to 1 s window.

**What was found about the dataset.**
- The spike times are not on the clock of the trials table and the files hold no synchronisation signal; the lag
  that best aligns them drifts by tens of seconds within each session. None of the paper's neural findings can be
  tested.
- The dataset holds 2 of the paper's 19 animals; no wall positions, whisker tracking, touch times or gait.
- In 489 the trials table rows 1000 to 1610 repeat one row; 256 of their 305 turns go right although all are
  labelled OL_L. In both sessions some 60 OL rows have stop times before their start times, and their sample
  timestamps restart at the previous trial's start.
- Unit counts (61 and 43) are below Table S1 (65 and 48), and 489's FS and RS counts (30 and 31) differ from Fig. 5B
  (43 and 22). Unit 489101 is FS by its waveform duration (0.2 ms) and labelled Inhibitory, where the paper calls it
  RS. 503 has three units labelled WM.
- The age of 489 is P63D in the file and 68 days from its date of birth; 503 runs at 13.1 cm/s in 27.7% of trials
  where Table S1 gives 16.1 cm/s and 52.8%.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")